In [1]:
from tutorials.utils.tutorial_utils import setup_notebook

setup_notebook()

# (Optional) Increase notebook width for all embedded cells to display properly
from IPython.core.display import display, HTML

display(HTML("<style>.output_result { max-width:100% !important; }</style>"))
display(HTML("<style>.container { width:100% !important; }</style>"))

/tmp/ipykernel_1168809/2402648777.py:6: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [2]:
# Useful imports
import os
from pathlib import Path
import tempfile

import hydra

from typing import List, Type

import numpy as np
import numpy.typing as npt

from nuplan.common.actor_state.ego_state import DynamicCarState, EgoState
from nuplan.common.actor_state.state_representation import StateSE2, StateVector2D, TimePoint
from nuplan.common.actor_state.vehicle_parameters import get_pacifica_parameters, VehicleParameters
from nuplan.planning.simulation.observation.observation_type import DetectionsTracks, Observation
from nuplan.planning.simulation.planner.abstract_planner import AbstractPlanner, PlannerInitialization, PlannerInput
from nuplan.planning.simulation.trajectory.interpolated_trajectory import  InterpolatedTrajectory
from nuplan.planning.simulation.trajectory.abstract_trajectory import AbstractTrajectory
from nuplan.planning.simulation.controller.motion_model.kinematic_bicycle import KinematicBicycleModel




In [3]:
from tutorials.utils.tutorial_utils import construct_simulation_hydra_paths

# Location of paths with all simulation configs
BASE_CONFIG_PATH = os.path.join(os.getenv('NUPLAN_TUTORIAL_PATH', ''), '../nuplan/planning/script')
simulation_hydra_paths = construct_simulation_hydra_paths(BASE_CONFIG_PATH)

# Create a temporary directory to store the simulation artifacts
SAVE_DIR = tempfile.mkdtemp()

# Select simulation parameters
EGO_CONTROLLER = 'perfect_tracking_controller'  # [log_play_back_controller, perfect_tracking_controller]
OBSERVATION = 'idm_agents_observation'  # [box_observation, idm_agents_observation, lidar_pc_observation]
DATASET_PARAMS = [
    'scenario_builder=nuplan_mini',  # use nuplan mini database (2.5h of 8 autolabeled logs in Las Vegas)
    'scenario_filter=one_continuous_log',  # simulate only one log
    "scenario_filter.log_names=['2021.06.09.11.54.15_veh-12_04366_04810']",
    'scenario_filter.limit_total_scenarios=2',  # use 2 total scenarios
]

# Initialize configuration management system
hydra.core.global_hydra.GlobalHydra.instance().clear()  # reinitialize hydra if already initialized
hydra.initialize(config_path=simulation_hydra_paths.config_path)

# Compose the configuration
cfg = hydra.compose(config_name=simulation_hydra_paths.config_name, overrides=[
    f'group={SAVE_DIR}',
    f'experiment_name=planner_tutorial',
    f'job_name=planner_tutorial',
    'experiment=${experiment_name}/${job_name}',
    'worker=sequential',
    f'ego_controller={EGO_CONTROLLER}',
    f'observation={OBSERVATION}',
    f'hydra.searchpath=[{simulation_hydra_paths.common_dir}, {simulation_hydra_paths.experiment_dir}]',
    'output_dir=${group}/${experiment}',
    *DATASET_PARAMS,
])

In [4]:
from nuplan.planning.script.run_simulation import run_simulation as main_simulation
from nuplan.planning.simulation.planner.smpc_planner import SMPCPlanner

ev_noise_std=[0.01,0.1]
tv_noise_std=[0.1, 0.1]

planner = SMPCPlanner(ev_noise_std=ev_noise_std, tv_noise_std=tv_noise_std)

# Run the simulation loop (real-time visualization not yet supported, see next section for visualization)
main_simulation(cfg, planner)

Global seed set to 0
INFO:nuplan.planning.script.builders.main_callback_builder:Building MultiMainCallback...
INFO:nuplan.planning.script.builders.main_callback_builder:Building MultiMainCallback: 4...DONE!


2024-08-17 16:13:20,489 INFO {/home/mpc/nuplan-devkit/nuplan/planning/script/builders/worker_pool_builder.py:19}  Building WorkerPool...
2024-08-17 16:13:20,492 INFO {/home/mpc/nuplan-devkit/nuplan/planning/utils/multithreading/worker_pool.py:101}  Worker: Sequential
2024-08-17 16:13:20,492 INFO {/home/mpc/nuplan-devkit/nuplan/planning/utils/multithreading/worker_pool.py:102}  Number of nodes: 1
Number of CPUs per node: 1
Number of GPUs per node: 0
Number of threads across all nodes: 1
2024-08-17 16:13:20,492 INFO {/home/mpc/nuplan-devkit/nuplan/planning/script/builders/worker_pool_builder.py:27}  Building WorkerPool...DONE!
2024-08-17 16:13:20,492 INFO {/home/mpc/nuplan-devkit/nuplan/planning/script/builders/folder_builder.py:32}  Building experiment folders...
2024-08-17 16:13:20,492 INFO {/home/mpc/nuplan-devkit/nuplan/planning/script/builders/folder_builder.py:35}  

	Folder where all results are stored: /tmp/tmpie1hw1dc/planner_tutorial/planner_tutorial

2024-08-17 16:13:20,494 IN

2024-08-17 16:13:23,783 WARNING {/home/mpc/nuplan-devkit/nuplan/planning/simulation/planner/smpc_planner.py:276}  IDMPlanner could not find valid path to the target roadblock. Using longest route found instead


/home/mpc/miniconda3/envs/nuplan/lib/python3.9/site-packages/shapely/linear.py:88: RuntimeWarning: invalid value encountered in line_locate_point
  return lib.line_locate_point(line, other)


2024-08-17 16:13:26,572 WARNING {/home/mpc/nuplan-devkit/nuplan/planning/simulation/planner/smpc_planner.py:276}  IDMPlanner could not find valid path to the target roadblock. Using longest route found instead


...

GETTING UPDATE DICT...
> /home/mpc/nuplan-devkit/nuplan/planning/simulation/planner/utils/smpc_utils.py(59)get_preds()
     57             if t < params['N']:
     58                 u_tvs[j][:,t] = agent._u_prev
---> 59             if t == 0:
     60                 agent_paths.append(agent._path)
     61                 tv_lengths.append(agent.length)



Traceback (most recent call last):
  File "/home/mpc/nuplan-devkit/nuplan/planning/simulation/runner/executor.py", line 27, in run_simulation
    return sim_runner.run()
  File "/home/mpc/nuplan-devkit/nuplan/planning/simulation/runner/simulations_runner.py", line 119, in run
    trajectory = self.planner.compute_trajectory(planner_input,preds)
  File "/home/mpc/nuplan-devkit/nuplan/planning/simulation/planner/abstract_planner.py", line 108, in compute_trajectory
    trajectory = self.compute_planner_trajectory(current_input, preds)
  File "/home/mpc/nuplan-devkit/nuplan/planning/simulation/planner/abstract_planner.py", line 108, in compute_trajectory
    trajectory = self.compute_planner_trajectory(current_input, preds)
  File "/home/mpc/nuplan-devkit/nuplan/planning/simulation/planner/smpc_planner.py", line 166, in compute_planner_trajectory
    update_dict = self.get_update_dict(current_input, preds)
  File "/home/mpc/nuplan-devkit/nuplan/planning/simulation/planner/smpc_planner.py"

2024-08-17 16:17:37,684 WARNING {/home/mpc/nuplan-devkit/nuplan/planning/simulation/runner/executor.py:27}  ----------- Simulation failed: with the following trace:
2024-08-17 16:17:37,685 WARNING {/home/mpc/nuplan-devkit/nuplan/planning/simulation/runner/executor.py:27}  Simulation failed with error:
 
2024-08-17 16:17:37,685 WARNING {/home/mpc/nuplan-devkit/nuplan/planning/simulation/runner/executor.py:27}  
Failed simulation [log,token]:
 [2021.06.09.11.54.15_veh-12_04366_04810, 7f0d0f8b4a3453cb]

2024-08-17 16:17:37,685 WARNING {/home/mpc/nuplan-devkit/nuplan/planning/simulation/runner/executor.py:27}  ----------- Simulation failed!
2024-08-17 16:17:37,686 WARNING {/home/mpc/nuplan-devkit/nuplan/planning/simulation/runner/executor.py:83}  Failed Simulation.
 'Traceback (most recent call last):
  File "/home/mpc/nuplan-devkit/nuplan/planning/simulation/runner/executor.py", line 27, in run_simulation
    return sim_runner.run()
  File "/home/mpc/nuplan-devkit/nuplan/planning/simulati

Rendering histograms: 0it [00:00, ?it/s]

2024-08-17 16:17:37,728 INFO {/home/mpc/nuplan-devkit/nuplan/planning/simulation/main_callback/metric_summary_callback.py:344}  Metric summary: 00:00:00 [HH:MM:SS]
2024-08-17 16:17:37,729 INFO {/home/mpc/nuplan-devkit/nuplan/planning/script/run_simulation.py:79}  Finished running simulation!


In [5]:
# Get nuBoard simulation file for visualization later on
simulation_file = [str(file) for file in Path(cfg.output_dir).iterdir() if file.is_file() and file.suffix == '.nuboard']

# Nuboard

In [6]:
from tutorials.utils.tutorial_utils import construct_nuboard_hydra_paths

# Location of paths with all nuBoard configs
nuboard_hydra_paths = construct_nuboard_hydra_paths(BASE_CONFIG_PATH)

# Initialize configuration management system
hydra.core.global_hydra.GlobalHydra.instance().clear()  # reinitialize hydra if already initialized
hydra.initialize(config_path=nuboard_hydra_paths.config_path)

# Compose the configuration
cfg = hydra.compose(config_name=nuboard_hydra_paths.config_name, overrides=[
    'scenario_builder=nuplan_mini',  # set the database (same as simulation) used to fetch data for visualization
    f'simulation_path={simulation_file}',  # nuboard file path, if left empty the user can open the file inside nuBoard
    f'hydra.searchpath=[{nuboard_hydra_paths.common_dir}, {nuboard_hydra_paths.experiment_dir}]',
])

from nuplan.planning.script.run_nuboard import main as main_nuboard

# Run nuBoard
main_nuboard(cfg)

INFO:bokeh.server.server:Starting Bokeh server version 2.4.3 (running on Tornado 6.4)
INFO:bokeh.server.tornado:User authentication hooks NOT provided (default user enabled)


2024-08-17 16:17:38,455 INFO {/home/mpc/nuplan-devkit/nuplan/planning/script/builders/scenario_building_builder.py:18}  Building AbstractScenarioBuilder...
2024-08-17 16:17:38,470 INFO {/home/mpc/nuplan-devkit/nuplan/planning/script/builders/scenario_building_builder.py:21}  Building AbstractScenarioBuilder...DONE!
2024-08-17 16:17:38,473 INFO {/home/mpc/nuplan-devkit/nuplan/planning/nuboard/nuboard.py:84}  Opening Bokeh application on http://localhost:5006/
2024-08-17 16:17:38,473 INFO {/home/mpc/nuplan-devkit/nuplan/planning/nuboard/nuboard.py:85}  Async rendering is set to: True
2024-08-17 16:17:43,893 INFO {/home/mpc/nuplan-devkit/nuplan/planning/nuboard/base/simulation_tile.py:172}  Minimum frame time=0.017 s
2024-08-17 16:17:43,901 INFO {/home/mpc/nuplan-devkit/nuplan/planning/nuboard/tabs/scenario_tab.py:485}  Rending scenario plot takes 0.0014 seconds.
2024-08-17 16:17:44,091 INFO {/home/mpc/miniconda3/envs/nuplan/lib/python3.9/site-packages/tornado/web.py:2348}  200 GET / (127

INFO:tornado.access:200 GET / (127.0.0.1) 248.16ms
